In [1]:
import ctadata
from astropy.io import fits
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord

In [12]:
# TODO: change if necessary obj_data_path
head_data_path = "../cta_dc_data/"
irfs_data_path = head_data_path + "irfs/"
obj_data_path = head_data_path + "mrk_501/"
obj_data_path_40deg = obj_data_path + "40deg/"
obj_data_path_20deg = obj_data_path + "20deg/"

In [13]:
!mkdir -p {irfs_data_path}
!mkdir -p {obj_data_path_40deg}
!mkdir -p {obj_data_path_20deg}

In [14]:
ctadata.fetch_and_save_file(f'pnfs/cta.cscs.ch/cta/sdc-internal-data/obs-index.fits.gz')
!mv obs-index.fits.gz {head_data_path}

In [15]:
# TODO: change if necessary obj_coord
obs_list = fits.open(head_data_path + "obs-index.fits.gz")[1]
obs_ra = np.array(obs_list.data["RA_PNT"])
obs_dec = np.array(obs_list.data["DEC_PNT"])
obs_coord = SkyCoord(ra=obs_ra*u.deg, dec=obs_dec*u.deg, frame='icrs')

obj_coord = SkyCoord.from_name("Mrk 501")

In [16]:
max_separation = 2
obj_obs_mask = obs_coord.separation(obj_coord).degree < max_separation

In [17]:
irf_files = []
irf_files_set = set()

for i, file in enumerate(obs_list.data["IRF_FILENAME"][obj_obs_mask]):
    irf_files.append(file)
    irf_files_set.add(file)

for file in irf_files_set:
    print(str(file))

Prod5-North-20deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz
Prod5-North-40deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz


In [8]:
# If the link has not changed, this should download directly the IRF compressed folder from zenodo,
# and decompress it to retrieve the two files needed.
# TODO: The archives names are hardcoded, check that they are indeed the correct ones (North 20 and 40 deg).
!mkdir -p tmp && cd tmp && \
wget https://zenodo.org/records/5499840/files/cta-prod5-zenodo-fitsonly-v0.1.zip?download=1 -O cta-prod5-zenodo-fitsonly-v0.1.zip && \
unzip cta-prod5-zenodo-fitsonly-v0.1.zip && \
tar -xzf fits/CTA-Performance-prod5-v0.1-North-20deg.FITS.tar.gz && \
tar -xzf fits/CTA-Performance-prod5-v0.1-North-40deg.FITS.tar.gz

for irf_file in irf_files_set:
    !mv tmp/{irf_file} {irfs_data_path}

!rm -r tmp

--2026-03-30 12:08:50--  https://zenodo.org/records/5499840/files/cta-prod5-zenodo-fitsonly-v0.1.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 44065624 (42M) [application/octet-stream]
Saving to: ‘cta-prod5-zenodo-fitsonly-v0.1.zip’

cta-prod5-zenodo-fi 100%[===================>]  42.02M  7.37MB/s    in 5.7s    

2026-03-30 12:08:56 (7.33 MB/s) - ‘cta-prod5-zenodo-fitsonly-v0.1.zip’ saved [44065624/44065624]

Archive:  cta-prod5-zenodo-fitsonly-v0.1.zip
replace LICENSE? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
mv: cannot stat 'tmp/Prod5-North-20deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz': No such file or directory
mv: cannot stat 'tmp/Prod5-North-40deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz': No such file or directory


In [18]:
for i, obs_id in enumerate(obs_list.data["OBS_ID"][obj_obs_mask]):
    ctadata.fetch_and_save_file(f'pnfs/cta.cscs.ch/cta/sdc-internal-data/products_SDC/Events/North/events_{obs_id}.fits')
    if irf_files[i] == "Prod5-North-40deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz":
        !mv events_{obs_id}.fits {obj_data_path_40deg}
    elif irf_files[i] == "Prod5-North-20deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz":
        !mv events_{obs_id}.fits {obj_data_path_20deg}